In [10]:
from article_utils.util import read_jsonl

reviews_df = read_jsonl('admin.jsonl.gz')
print(reviews_df.columns)
reviews_df

Index(['Comments', 'abstract', 'accuracy', 'civic.id', 'claim.flat',
       'document.pmcid', 'document.pmid', 'evidence.flat', 'example.id',
       'filepath', 'generated_text', 'has_content', 'id', 'label', 'partition',
       'predicted_label_name', 'text'],
      dtype='object')


,Comments,abstract,accuracy,civic.id,claim.flat,document.pmcid,document.pmid,evidence.flat,example.id,filepath,generated_text,has_content,id,label,partition,predicted_label_name,text
0,[No results or conclusions were reported that ...,False,0.00,[10047],molecular profile\tevidence type\tsignificance...,PMC2118656,17646409,"he genesis of breast cancer, medulloblastoma, ...",[claim:644210_ev:672679_REFUTES],results claim_verification fine-tuning config-...,label: NEI. The variant (S2514P) was not menti...,True,[claim:644210_ev:672679_REFUTES_r0],"[Response - Irrelevant, Response - contradicts...",train,NEI,"<div class=""example""><div class=""question""><h2..."
1,"[This is a mutation of the NOTCH1 receptor., t...",False,0.00,[10047],molecular profile\tevidence type\tsignificance...,PMC2118656,17646409,"he genesis of breast cancer, medulloblastoma, ...",[claim:644210_ev:672679_REFUTES],results claim_verification fine-tuning config-...,label: NEI. There are no data presented. \nlab...,True,[claim:644210_ev:672679_REFUTES_r0],"[Response - Irrelevant, Response - Poor gramma...",train,NEI,"<div class=""example""><div class=""question""><h2..."
2,[The S2514P mutation in the evidence is descri...,False,0.00,[10047],molecular profile\tevidence type\tsignificance...,PMC2118656,17646409,"he genesis of breast cancer, medulloblastoma, ...",[claim:644210_ev:672679_REFUTES],results claim_verification fine-tuning config-...,label: SUPPORTS. The S2514P mutation in the ev...,True,[claim:644210_ev:672679_REFUTES_r0],"[Response - Major Errors (logic errors, invali...",train,SUPPORTS,"<div class=""example""><div class=""question""><h2..."
3,[],False,0.00,[10047],molecular profile\tevidence type\tsignificance...,PMC2118656,17646409,"he genesis of breast cancer, medulloblastoma, ...",[claim:644210_ev:672679_REFUTES],results claim_verification fine-tuning config-...,label: SUPPORTS. This seems to involve NOTCH g...,True,[claim:644210_ev:672679_REFUTES_r0],"[Response - Irrelevant, Response - Uses materi...",train,SUPPORTS,"<div class=""example""><div class=""question""><h2..."
4,[REFUTES. Binding was restored when the author...,False,0.00,[10046],molecular profile\tevidence type\tsignificance...,PMC2118656,17646409,"he genesis of breast cancer, medulloblastoma, ...",[claim:644209_ev:672775_REFUTES],results claim_verification fine-tuning config-...,label: GAIN of function. These experiments sup...,True,[claim:644209_ev:672775_REFUTES_r0],"[Response - Major Errors (logic errors, invali...",train,GAIN,"<div class=""example""><div class=""question""><h2..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
391,[],False,0.25,[1986],molecular profile\tevidence type\tsignificance...,PMC3203921,22046346,lapatinib and alternative ERBB2 inhibitors. Bo...,[claim:638473_ev:681398_SUPPORTS],results claim_verification fine-tuning config-...,label: SUPPORTS. ERBB2 T862A mutation found in...,False,[claim:638473_ev:681398_SUPPORTS_r0],[Gold Label Borderline],train,SUPPORTS,"<div class=""example""><div class=""question""><h2..."
392,"[However, it does say that the mutation result...",False,0.25,[10121],molecular profile\tevidence type\tsignificance...,PMC3293822,22403669,tion of the phosphatidylinositol 3-kinase (PI3...,[claim:644283_ev:672573_SUPPORTS],results claim_verification fine-tuning config-...,label: NEI. The evidence describes the mutatio...,True,[claim:644283_ev:672573_SUPPORTS_r0],[Response - contradicts/wrong about evidence],train,NEI,"<div class=""example""><div class=""question""><h2..."
393,[],False,0.25,[10121],molecular profile\tevidence type\tsignificance...,PMC3293822,22403669,tion of the phosphatidylinositol 3-kinase (PI3...,[claim:644283_ev:672573_SUPPORTS],results claim_verification fine-tuning config-...,label: NEI. The evidence is not clear enough t...,True,[claim:644283_ev:672573_SUPPORTS_r0],"[Response - Major Errors (logic errors, invali...",train,NEI,"<div class=""example""><div class=""question""><h2..."
394,[],False,0

In [11]:
from article_utils.util import dump_jsonl



# drop any comments that are selected spans from the answer. We only care about the new gold labels and explanations entered into the comments here
def exclude_spans(row):
    matches = []

    for comment in row.Comments:
        comment = comment.strip()
        pos = row.generated_text.find(comment.strip())
        if pos == -1:
            matches.append(comment)
    if matches:
        assert len(matches) == 1
        return matches[0]
    return ''


reviews_df['tags'] = reviews_df.label
reviews_df['reason'] = reviews_df.apply(exclude_spans, axis=1)
reviews_df = reviews_df[reviews_df.reason.fillna('') != '']
reviews_df['gold_label_name'] = reviews_df.reason.str.split('.').str[0].str.strip()
reviews_df['reason'] = reviews_df.reason.str.split('.').str[1].str.strip()
reviews_df['filepath'] = reviews_df.filepath.apply(lambda p: '/'.join(p.split(' ')))

for f in sorted(reviews_df.filepath.unique()):
    print(f)
reviews_df = reviews_df.copy().reset_index(drop=True)[['claim.flat', 'evidence.flat', 'document.pmid', 'filepath', 'reason', 'gold_label_name']].drop_duplicates()
reviews_df

results/claim_verification/fine-tuning/config-8/completion/civicfact-2025.01.14/chat-default/prompts.reason.0-shot.train.jsonl/BioMistral__BioMistral-7B-DARE/evaluations/civicfactpairs-grand-bat-L5000-2024.12.22/chat-default/gen-1/prompts.reason.0-shot.dev.processed.jsonl.gz
results/claim_verification/fine-tuning/config-8/completion/civicfact-2025.01.14/chat-default/prompts.reason.0-shot.train.jsonl/BioMistral__BioMistral-7B-DARE/evaluations/civicfactpairs-grand-bat-L5000-2024.12.22/chat-default/gen-1/prompts.reason.0-shot.train.processed.jsonl.gz
results/claim_verification/fine-tuning/config-8/completion/civicfact-2025.01.14/chat-default/prompts.reason.0-shot.train.jsonl/BioMistral__BioMistral-7B-DARE/evaluations/civicfactpairs-grand-bat-L5000-2024.12.22/chat-default/gen-2/prompts.reason.0-shot.dev.processed.jsonl.gz
results/claim_verification/fine-tuning/config-8/completion/civicfact-2025.01.14/chat-default/prompts.reason.0-shot.train.jsonl/BioMistral__BioMistral-7B-DARE/evaluations/

/var/tmp/ipykernel_8092/3223055409.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reviews_df['gold_label_name'] = reviews_df.reason.str.split('.').str[0].str.strip()
/var/tmp/ipykernel_8092/3223055409.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reviews_df['reason'] = reviews_df.reason.str.split('.').str[1].str.strip()
/var/tmp/ipykernel_8092/3223055409.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = v

,claim.flat,evidence.flat,document.pmid,filepath,reason,gold_label_name
0,molecular profile\tevidence type\tsignificance...,"he genesis of breast cancer, medulloblastoma, ...",17646409,results/claim_verification/fine-tuning/config-...,The authors found that NOTCH binding to FBW7 w...,REFUTES
1,molecular profile\tevidence type\tsignificance...,"he genesis of breast cancer, medulloblastoma, ...",17646409,results/claim_verification/fine-tuning/config-...,Binding was restored when the authors mutated ...,REFUTES
2,molecular profile\tevidence type\tsignificance...,Prognostic and predictive value of common muta...,19603024,results/claim_verification/fine-tuning/config-...,Irinotecan is one of several current treatment...,REFUTES
3,molecular profile\tevidence type\tsignificance...,ficantly lower progression-free survival (PFS:...,19603024,results/claim_verification/fine-tuning/config-...,BRAF mutations in colorectal cancer and their ...,NEI
4,molecular profile\tevidence type\tsignificance...,ficantly lower progression-free survival (PFS:...,19603024,results/claim_verification/fine-tuning/config-...,Oxaliplatin is a current treatment regimen for...,REFUTES
5,molecular profile\tevidence type\tsignificance...,but not for dasatinib (DeltaDeltaGbind = +0.3...,19671763,results/claim_verification/fine-tuning/config-...,The evidence discusses KIT mutation response t...,NEI
6,molecular profile\tevidence type\tsignificance...,Differential Sensitivity of ERBB2 Kinase Domai...,22046346,results/claim_verification/fine-tuning/config-...,"According to the evidence, a cell line with th...",SUPPORTS
7,molecular profile\tevidence type\tsignificance...,and resistance to treatment with nucleoside a...,23377183,results/claim_verification/fine-tuning/config-...,"In contrast the to claim, the evidence reports...",REFUTES
8,molecular profile\tevidence type\tsignificance...,and resistance to treatment with nucleoside a...,23377183,results/claim_verification/fine-tuning/config-...,"The R367Q was tested in Reh cells, which is a ...",REFUTES
9,molecular profile\tevidence type\tsignificance...,"human cancers, and can be triggered by a vari...",24088382,results/claim_verification/fine-tuning/config-...,PTEN null cell lines were treated with capivas...,NEI


In [12]:
import os

llm_df = read_jsonl(*[os.path.join('/projects/creisle_prj/git/llm_testing', f) for f in  reviews_df.filepath.unique().tolist()], add_filename=True)
print(llm_df.columns)
llm_df

Index(['assignment.round', 'chat_template', 'civic.evidenceDirection',
       'civic.flagged', 'civic.id', 'civic.status', 'claim.flat',
       'claim.generated', 'claim.hash', 'claim.id', 'claim.source',
       'document.doi', 'document.license', 'document.pmcid', 'document.pmid',
       'document.title', 'evidence.flat', 'example.id', 'examples',
       'generated', 'generated_text', 'gold_label_name', 'id', 'label',
       'label.tokens', 'model', 'predicted_label_name', 'prompt',
       'prompt.chat', 'prompt.eval', 'prompt.response_template',
       'prompt.tokens', 'prompt.train', 'prompt_examples', 'reason',
       'reason.id', 'redundancy', 'summary', 'template', 'filename'],
      dtype='object')


,assignment.round,chat_template,civic.evidenceDirection,civic.flagged,civic.id,civic.status,claim.flat,claim.generated,claim.hash,claim.id,...,prompt.response_template,prompt.tokens,prompt.train,prompt_examples,reason,reason.id,redundancy,summary,template,filename
0,1,llama-3-chat,SUPPORTS,0.0,11004,SUBMITTED,molecular profile\tevidence type\tsignificance...,False,645056,claim:645056,...,<|start_header_id|>assistant<|end_header_id|>\n\n,1309,<|begin_of_text|><|start_header_id|>user<|end_...,[],,0,1,170 cases of IDH-wildtype GBM were screened us...,reason,/projects/creisle_prj/git/llm_testing/results/...
1,1,llama-3-chat,SUPPORTS,0.0,6567,ACCEPTED,molecular profile\tevidence type\tsignificance...,False,690719,claim:690719,...,<|start_header_id|>assistant<|end_header_id|>\n\n,1072,<|begin_of_text|><|start_header_id|>user<|end_...,[],,0,1,This was a Phase 1/2 study of 55 patients from...,reason,/projects/creisle_prj/git/llm_testing/results/...
2,1,llama-3-chat,SUPPORTS,0.0,5507,ACCEPTED,molecular profile\tevidence type\tsignificance...,False,640266,claim:640266,...,<|start_header_id|>assistant<|end_header_id|>\n\n,1316,<|begin_of_text|><|start_header_id|>user<|end_...,[],,0,1,A meta analysis of 60 research studies involvi...,reason,/projects/creisle_prj/git/llm_testing/results/...
3,1,llama-3-chat,SUPPORTS,0.0,1103,ACCEPTED,molecular profile\tevidence type\tsignificance...,False,637700,claim:637700,...,<|start_header_id|>assistant<|end_header_id|>\n\n,1274,<|begin_of_text|><|start_header_id|>user<|end_...,[],,0,1,This preclinical study establishes that the p3...,reason,/projects/creisle_prj/git/llm_testing/results/...
4,1,llama-3-chat,DOES_NOT_SUPPORT,0.0,10016,ACCEPTED,molecular profile\tevidence type\tsignificance...,False,644180,claim:644180,...,<|start_header_id|>assistant<|end_header_id|>\n\n,1290,<|begin_of_text|><|start_header_id|>user<|end_...,[],,0,1,ALK fusion positive NSCLC cell lines with high...,reason,/projects/creisle_prj/git/llm_testing/results/...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
904,3,default,,NaN,,,molecular profile\tevidence type\tsignificance...,True,3695,claim:3695,...,[/INST],1870,"<s>[INST] Given a claim and some evidence, det...",[],The table says that the functional status is l...,0,3,,reason,/projects/creisle_prj/git/llm_testing/results/...
905,3,default,,NaN,,,molecular profile\tevidence type\tsignificance...,True,3695,claim:3695,...,[/INST],1870,"<s>[INST] Given a claim and some evidence, det...",[],The evidence lists the function status of the ...,1,3,,reason,/projects/creisle_prj/git/llm_testing/results/...
906,3,default,,NaN,,,molecular profile\tevidence type\tsignificance...,True,5081,claim:5081,...,[/INST],402,"<s>[INST] Given a claim and some evidence, det...",[],,0,3,,reason,/projects/creisle_prj/git/llm_testing/results/...
907,3,default,,NaN,,,molecular profile\tevidence type\tsignificance...,True,399,claim:399,...,[/INST],598,"<s>[INST] Given a claim and some evidence, det...",[],,0,3,,reason,/projects/creisle_prj/git/llm_testing/results/...


In [13]:
import re
import pandas as pd
import json

def parse_filepath_metadata(filename):
    subpatterns = {
        "train_data": r"(?P<train_dataset>\w+)-(?P<train_data_version>[^/]+)",
        "eval_data": r"(?P<eval_dataset>(civicfact|[^/-]+))(?P<eval_dataset_claim_format>nl|isa)?(-(?P<ret_model>(\w+-)*\w+)-(?P<ret_length>L\d+))?-(?P<data_version>[^/-]+)",
    }
    patterns = [
        (
            r"results/(?P<task>claim_verification)/fine-tuning/config-(?P<config>\d+)/(?P<method>seqclass)/"
            + subpatterns["train_data"]
            + r"/(?P<model_dir>[^/]+)/evaluations/"
            + subpatterns["eval_data"]
            + r"/data.(?P<partition>\w+).processed.((label_stability|label_accuracy)\.)?jsonl(\.gz)?$"
        ),
        (
            r"results/(?P<task>claim_verification)/fine-tuning/config-(?P<config>\d+)/(?P<method>\w+)/"
            + subpatterns["train_data"]
            + r"/chat-(?P<train_chat_template>\w+)/[^/]+/(?P<model_dir>[^/]+)/evaluations/"
            + subpatterns["eval_data"]
            + r"/chat-(?P<chat_template>\w+)/(?P<generation_config>gen-\d+)/prompts.(?P<prompt_template>\w+).(?P<examples>\d+)-shot.(?P<partition>\w+).processed.((label_stability|label_accuracy)\.)?jsonl(\.gz)?$"
        ),
        (
            r"results/(?P<task>claim_verification)/evaluations(?P<quantized>_noquant)?/(?P<model_dir>[^/]+)/"
            + subpatterns["eval_data"]
            + r"/chat-(?P<chat_template>\w+)/(?P<generation_config>gen-\d+)/prompts.(?P<prompt_template>\w+).(?P<examples>\d+)-shot.(?P<partition>\w+).processed.((label_stability|label_accuracy)\.)?jsonl(\.gz)?$"
        ),
        (
            r"results/(?P<task>evidence_retrieval)/(?P<encoding>cross|bi)/fine-tuning/config-(?P<config>\d+)/"
            + subpatterns["train_data"]
            + r"/(?P<model_dir>[^/]+)"
        ),
        r"results/(?P<task>evidence_retrieval)/(?P<encoding>cross|bi)/evaluations/(?P<model_dir>[^/]+)",
    ]

    groups = None
    for patt in patterns:
        m = re.search(patt, filename)
        if m:
            groups = m.groupdict()
            break
    if groups is None:
        raise AssertionError(f"failed to match metadata for filename: {filename}")
    return dict(groups)


def collect_filepath_metadata(filename):

    metadata = parse_filepath_metadata(filename)
    if metadata.get("ret_model"):
        path_nicknames_file = "/projects/creisle_prj/git/llm_testing/data/pathnames.json"
        with open(path_nicknames_file, "r") as fh:
            path_nicknames = json.load(fh)
        ret_meta = parse_filepath_metadata(path_nicknames[metadata["ret_model"]])
        metadata.update({f"ret_{k}": v for k, v in ret_meta.items()})
    return metadata

meta_df = []

for filename in llm_df.filename.unique():
    meta = collect_filepath_metadata(filename)
    input_dataset = os.path.join('/projects/creisle_prj/git/llm_testing/data','-'.join([meta['eval_dataset'], meta['ret_model'], meta['ret_length'], meta['data_version']]), 'data.jsonl.gz')
    df = read_jsonl(input_dataset)
    for col, val in meta.items():
        if col.startswith('ret_'):
            df[col.replace('ret_', 'evidence.retrieval.')] = val
    df['filename'] = filename
    df['evidence.selection_method'] = 'ret_model'
    meta_df.append(df)


meta_df = pd.concat(meta_df)
drop_cols = [
    'reviews',
    'evidence.hash',
    'claim.hash',
    'chunks',
    'is_expected_chunk',
    'original_example',
    'conversation',
    'in_table',
    'model_name',
    'id',
    'example.id',
    'display_id',
    'tracking_id',
    'filename',
    'example',
    'gold_label_name',
    'generated'
]
meta_df = meta_df.drop(columns=set([c for c in meta_df.columns if c.startswith('expected_evidence') or c.startswith('hypothesis') or c.startswith('correct.')] + drop_cols) & set(meta_df.columns))

meta_df = meta_df.drop_duplicates([c for c in meta_df.columns if c not in ['evidence', 'claim']])

print(meta_df.columns)
print(meta_df.dtypes)
meta_df

Index(['civic.description', 'civic.evidenceDirection', 'civic.flagged',
       'civic.id', 'civic.status', 'claim', 'claim.flat', 'claim.generated',
       'claim.id', 'claim.source', 'document.doi', 'document.license',
       'document.pmcid', 'document.pmid', 'document.title', 'evidence',
       'evidence.flat', 'flagged', 'is_flagged', 'partition', 'pmid.partition',
       'evidence.retrieval.model', 'evidence.retrieval.length',
       'evidence.retrieval.task', 'evidence.retrieval.encoding',
       'evidence.retrieval.model_dir', 'evidence.selection_method'],
      dtype='object')
civic.description               object
civic.evidenceDirection         object
civic.flagged                   object
civic.id                        object
civic.status                    object
claim                           object
claim.flat                      object
claim.generated                   bool
claim.id                        object
claim.source                    object
document.doi      

,civic.description,civic.evidenceDirection,civic.flagged,civic.id,civic.status,claim,claim.flat,claim.generated,claim.id,claim.source,...,flagged,is_flagged,partition,pmid.partition,evidence.retrieval.model,evidence.retrieval.length,evidence.retrieval.task,evidence.retrieval.encoding,evidence.retrieval.model_dir,evidence.selection_method
0,170 cases of IDH-wildtype GBM were screened us...,SUPPORTS,False,11004,SUBMITTED,"[{'content': [['molecularProfile', 'EGFR::SEPT...",molecular profile\tevidence type\tsignificance...,False,claim:645056,civic,...,False,NaN,train,train,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model
1,This was a Phase 1/2 study of 55 patients from...,SUPPORTS,False,6567,ACCEPTED,"[{'content': [['molecularProfile', 'v::NTRK1 F...",molecular profile\tevidence type\tsignificance...,False,claim:690719,civic,...,False,NaN,train,train,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model
2,A meta analysis of 60 research studies involvi...,SUPPORTS,False,5507,ACCEPTED,"[{'content': [['molecularProfile', 'CD274 Expr...",molecular profile\tevidence type\tsignificance...,False,claim:640266,civic,...,False,NaN,train,train,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model
3,This preclinical study establishes that the p3...,SUPPORTS,False,1103,ACCEPTED,"[{'content': [['molecularProfile', 'CEBPA N-TE...",molecular profile\tevidence type\tsignificance...,False,claim:637700,civic,...,False,NaN,train,train,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model
4,ALK fusion positive NSCLC cell lines with high...,DOES_NOT_SUPPORT,False,10016,ACCEPTED,"[{'content': [['molecularProfile', 'ABCB1 Over...",molecular profile\tevidence type\tsignificance...,False,claim:644180,civic,...,False,NaN,train,train,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1205,,,NaN,,,"[{'content': [['molecularProfile', 'NF1 Mutati...",molecular profile\tevidence type\tsignificance...,False,claim:936,civic,...,NaN,NaN,dev,dev,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model
1206,,,NaN,,,"[{'content': [['molecularProfile', 'TP53 R249S...",molecular profile\tevidence type\tsignificance...,True,claim:3695,civic flipped significance,...,NaN,NaN,dev,dev,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model
1207,,,NaN,,,"[{'content': [['molecularProfile', 'SMARCB1 Lo...",molecular profile\tevidence type\tsignificance...,True,claim:5081,neutral-bot,...,NaN,NaN,dev,dev,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model
1208,,,NaN,,,"[{'content': [['molecularProfile', 'GNAS c.393...",molecular profile\tevidence type\tsignificance...,True,claim:399,civic flipped significance,...,NaN,NaN,dev,dev,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model


In [14]:
print('shared columns:', sorted(set(reviews_df.columns) & set(meta_df.columns)))
print('llm columns:', sorted(set(llm_df.columns) - set(reviews_df.columns)))
print('reviews_df columns:', sorted(set(reviews_df.columns) - set(meta_df.columns)))


shared columns: ['claim.flat', 'document.pmid', 'evidence.flat']
llm columns: ['assignment.round', 'chat_template', 'civic.evidenceDirection', 'civic.flagged', 'civic.id', 'civic.status', 'claim.generated', 'claim.hash', 'claim.id', 'claim.source', 'document.doi', 'document.license', 'document.pmcid', 'document.title', 'example.id', 'examples', 'filename', 'generated', 'generated_text', 'id', 'label', 'label.tokens', 'model', 'predicted_label_name', 'prompt', 'prompt.chat', 'prompt.eval', 'prompt.response_template', 'prompt.tokens', 'prompt.train', 'prompt_examples', 'reason.id', 'redundancy', 'summary', 'template']
reviews_df columns: ['filepath', 'gold_label_name', 'reason']


In [15]:
# add the retrieval metadata to the input reviews df
size_before = reviews_df.shape[0]

reviews_df['document.pmid'] = reviews_df['document.pmid'].astype(str)
df = reviews_df[['document.pmid', 'reason', 'gold_label_name', 'claim.flat', 'evidence.flat']].drop_duplicates().merge(meta_df, on=['claim.flat', 'evidence.flat', 'document.pmid'])
print('reviews df before/after', size_before, df.shape[0])

for grouping_cols, subset_df in df.groupby(['claim.flat', 'evidence.flat', 'document.pmid']):
    diffs = {}
    for _, row in subset_df.iterrows():
        for col in dict(row):
            if not isinstance(row[col], list):
                diffs.setdefault(col, set()).add(row[col])

    for col in diffs:
        if len(diffs[col]) > 1:
            print(col, diffs[col])

assert size_before == df.shape[0]
df

reviews df before/after 40 40


,document.pmid,reason,gold_label_name,claim.flat,evidence.flat,civic.description,civic.evidenceDirection,civic.flagged,civic.id,civic.status,...,flagged,is_flagged,partition,pmid.partition,evidence.retrieval.model,evidence.retrieval.length,evidence.retrieval.task,evidence.retrieval.encoding,evidence.retrieval.model_dir,evidence.selection_method
0,17646409,The authors found that NOTCH binding to FBW7 w...,REFUTES,molecular profile\tevidence type\tsignificance...,"he genesis of breast cancer, medulloblastoma, ...",Seven T cell acute lymphoblastic leukemia (T-A...,DOES_NOT_SUPPORT,False,10047,SUBMITTED,...,False,NaN,train,train,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model
1,17646409,Binding was restored when the authors mutated ...,REFUTES,molecular profile\tevidence type\tsignificance...,"he genesis of breast cancer, medulloblastoma, ...",Seven T cell acute lymphoblastic leukemia (T-A...,DOES_NOT_SUPPORT,False,10046,SUBMITTED,...,False,NaN,train,train,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model
2,19603024,Irinotecan is one of several current treatment...,REFUTES,molecular profile\tevidence type\tsignificance...,Prognostic and predictive value of common muta...,This was a retrospective clinical study of 44 ...,DOES_NOT_SUPPORT,False,6299,ACCEPTED,...,False,NaN,train,train,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model
3,19603024,BRAF mutations in colorectal cancer and their ...,NEI,molecular profile\tevidence type\tsignificance...,ficantly lower progression-free survival (PFS:...,Of 100 metastatic colorectal cancer patients t...,SUPPORTS,False,6403,ACCEPTED,...,False,NaN,train,train,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model
4,19603024,Oxaliplatin is a current treatment regimen for...,REFUTES,molecular profile\tevidence type\tsignificance...,ficantly lower progression-free survival (PFS:...,This was a retrospective clinical study of 100...,DOES_NOT_SUPPORT,False,6293,ACCEPTED,...,False,NaN,train,train,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model
5,19671763,The evidence discusses KIT mutation response t...,NEI,molecular profile\tevidence type\tsignificance...,but not for dasatinib (DeltaDeltaGbind = +0.3...,"In an in vitro study, a melanoma cell line, A3...",SUPPORTS,False,3759,SUBMITTED,...,False,NaN,train,train,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model
6,22046346,"According to the evidence, a cell line with th...",SUPPORTS,molecular profile\tevidence type\tsignificance...,Differential Sensitivity of ERBB2 Kinase Domai...,Cell proliferation analysis of Ba/F3 cells exp...,SUPPORTS,False,1988,ACCEPTED,...,True,NaN,train,train,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model
7,23377183,"In contrast the to claim, the evidence reports...",REFUTES,molecular profile\tevidence type\tsignificance...,and resistance to treatment with nucleoside a...,Overexpression of NT5C2 S445F in Reh cells res...,DOES_NOT_SUPPORT,False,7817,ACCEPTED,...,False,NaN,train,train,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model
8,23377183,"The R367Q was tested in Reh cells, which is a ...",REFUTES,molecular profile\tevidence type\tsignificance...,and resistance to treatment with nucleoside a...,Lentiviral mediated overexpression of NT5C2 R3...,DOES_NOT_SUPPORT,False,7815,ACCEPTED,...,False,NaN,train,train,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model
9,24088382,PTEN null cell lines were treated with capivas...,NEI,molecular profile\tevidence type\tsignificance...,"human cancers, and can be triggered by a vari...",A patient derived mouse xenograft model was es...,SUPPORTS,False,1500,ACCEPTED,...,False,NaN,train,train,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model


In [16]:
# get the evidence.source / document.source column from the build file
build_df = read_jsonl('../../../builds/civicfact-2024.12.22/data.jsonl.gz')
build_df = build_df[build_df['evidence.source'] != 'civic']

build_df = build_df.fillna('')
build_df = build_df.groupby('document.pmid').agg({'evidence.source': 'unique', 'document.license': 'unique', 'document.pmcid': 'unique'}).reset_index()

def pick_source(row):
    sources = set(row['evidence.source']) - {''}
    if len(sources) == 1:
        return row['evidence.source'][0]
    if sources == {'pmc_author_fulltext', 'pmc'}:
        if len(set(row['document.license']) - {'none'}) == 0:
            return 'pmc_author_fulltext'
        else:
            return 'pmc'
    if sources == {'pubmed', 'pmc'}:
        if set(row['document.pmcid']) - {''} and len(set(row['document.license']) - {'none'}) > 0:
            return 'pmc'
        else:
            return 'pubmed'
    else:
        print(dict(row))
        raise NotImplementedError('idk what to do here')

build_df['evidence.source'] = build_df.apply(pick_source, axis=1)
build_df = build_df.drop(columns=['document.license', 'document.pmcid'])
build_df['evidence.source'].value_counts()


evidence.source
pubmed                 883
pmc                    474
pmc_author_fulltext    142
Name: count, dtype: int64

We want to create review objects which look something like this (example from previous review compilation)

```json
 {
    "comment": "evidence is not predictive based on this variant",
    "group_review_comments": [],     
    "group_review_label": "REFUTES",
    "group_review_round": "IAv2_error_review",
    "group_review_tags": [
        "Bot Generated",
        "Gold Label Incorrect"     
    ],     
    "group_review_validated": false,    
    "label": "NEI",
    "original_label": "NEI",     
    "reason": "",
    "round": "IAv2_civicfact2023.10.20",
    "username": "camgrisdale"
 }
```

In [17]:
def create_reviews(row):
    return [{
        'label': row['gold_label_name'],
        'reason': row['reason'],
        'round': '2025.02.24_xmodel_retrieval_error_review-2024.12.22',
        'username': 'creisle',
        'group_review_validated': False
    }]

df = df.merge(build_df, how='left')
df['document.content_source'] = df['evidence.source']
df['evidence.generated'] = True


df['reviews'] = df.apply(create_reviews, axis=1)
df

,document.pmid,reason,gold_label_name,claim.flat,evidence.flat,civic.description,civic.evidenceDirection,civic.flagged,civic.id,civic.status,...,evidence.retrieval.model,evidence.retrieval.length,evidence.retrieval.task,evidence.retrieval.encoding,evidence.retrieval.model_dir,evidence.selection_method,evidence.source,document.content_source,evidence.generated,reviews
0,17646409,The authors found that NOTCH binding to FBW7 w...,REFUTES,molecular profile\tevidence type\tsignificance...,"he genesis of breast cancer, medulloblastoma, ...",Seven T cell acute lymphoblastic leukemia (T-A...,DOES_NOT_SUPPORT,False,10047,SUBMITTED,...,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model,pmc,pmc,False,"[{'label': 'REFUTES', 'reason': 'The authors f..."
1,17646409,Binding was restored when the authors mutated ...,REFUTES,molecular profile\tevidence type\tsignificance...,"he genesis of breast cancer, medulloblastoma, ...",Seven T cell acute lymphoblastic leukemia (T-A...,DOES_NOT_SUPPORT,False,10046,SUBMITTED,...,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model,pmc,pmc,False,"[{'label': 'REFUTES', 'reason': 'Binding was r..."
2,19603024,Irinotecan is one of several current treatment...,REFUTES,molecular profile\tevidence type\tsignificance...,Prognostic and predictive value of common muta...,This was a retrospective clinical study of 44 ...,DOES_NOT_SUPPORT,False,6299,ACCEPTED,...,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model,pmc,pmc,False,"[{'label': 'REFUTES', 'reason': 'Irinotecan is..."
3,19603024,BRAF mutations in colorectal cancer and their ...,NEI,molecular profile\tevidence type\tsignificance...,ficantly lower progression-free survival (PFS:...,Of 100 metastatic colorectal cancer patients t...,SUPPORTS,False,6403,ACCEPTED,...,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model,pmc,pmc,False,"[{'label': 'NEI', 'reason': 'BRAF mutations in..."
4,19603024,Oxaliplatin is a current treatment regimen for...,REFUTES,molecular profile\tevidence type\tsignificance...,ficantly lower progression-free survival (PFS:...,This was a retrospective clinical study of 100...,DOES_NOT_SUPPORT,False,6293,ACCEPTED,...,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model,pmc,pmc,False,"[{'label': 'REFUTES', 'reason': 'Oxaliplatin i..."
5,19671763,The evidence discusses KIT mutation response t...,NEI,molecular profile\tevidence type\tsignificance...,but not for dasatinib (DeltaDeltaGbind = +0.3...,"In an in vitro study, a melanoma cell line, A3...",SUPPORTS,False,3759,SUBMITTED,...,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model,pmc,pmc,False,"[{'label': 'NEI', 'reason': 'The evidence disc..."
6,22046346,"According to the evidence, a cell line with th...",SUPPORTS,molecular profile\tevidence type\tsignificance...,Differential Sensitivity of ERBB2 Kinase Domai...,Cell proliferation analysis of Ba/F3 cells exp...,SUPPORTS,False,1988,ACCEPTED,...,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model,pmc,pmc,False,"[{'label': 'SUPPORTS', 'reason': 'According to..."
7,23377183,"In contrast the to claim, the evidence reports...",REFUTES,molecular profile\tevidence type\tsignificance...,and resistance to treatment with nucleoside a...,Overexpression of NT5C2 S445F in Reh cells res...,DOES_NOT_SUPPORT,False,7817,ACCEPTED,...,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model,pmc,pmc,False,"[{'label': 'REFUTES', 'reason': 'In contrast t..."
8,23377183,"The R367Q was tested in Reh cells, which is a ...",REFUTES,molecular profile\tevidence type\tsignificance...,and resistance to treatment with nucleoside a...,Lentiviral mediated overexpression of NT5C2 R3...,DOES_NOT_SUPPORT,False,7815,ACCEPTED,...,grand-bat,L5000,evidence_retrieval,cross,BAAI__bge-reranker-large,ret_model,pmc,pmc,False,"[{'label': 'REFUTES', 'reason': 'The R367Q was..."
9,24088382,PTEN null cell lin

In [18]:
from article_utils.util import dump_jsonl

dump_jsonl(df.drop(columns=[c for c in ['reason', 'label'] if c in df.columns]), 'reviews.jsonl')


Wherease the rows themselves should look something like this with the evidence containing the chunk.id(s) and other chunk metadata

```json
{
    "claim": [{"content": [ ], "meta": {"synonyms": [{"name": "", "synonyms": "", "type": ""}]}, "type": "table"}], 
    "claim.flat": "...", 
    "claim.generated": true, 
    "claim.id": "...", 
    "claim.source": "", 
    "document.doi": "", 
    "document.pmcid": "", 
    "document.pmid": "", 
    "document.title": "...", 
    "evidence": [
        {"content": "...", "meta": {"end": 0, "section": "", "start": 0}, "type": "unstructured"}, 
        {"content": "...", "meta": {"end": 0, "section": "", "start": 1884}, "type": "unstructured"}
    ], 
    "evidence.flat": "...", 
    "evidence.selection_method": "", 
    "evidence.source": "", 
    "gold_label_name": "", 
    "reviews": []
}

```